# Step 14.5 — Implement and Test One Transformer Block：第一次把 Attention、Norm、Residual、MLP 真正组装起来

Step 14.1–14.4 已分别准备好：

```text
Residual connection
RMSNorm / LayerNorm
Pre-Norm placement
Multi-Head Attention
Feed-Forward MLP
```

本节第一次把它们组装成一个完整的 **Pre-Norm Transformer Block**。

采用本课程当前最小设计：

```text
x
↓
RMSNorm
↓
Causal Multi-Head Attention
↓
Residual Add
↓
RMSNorm
↓
Feed-Forward MLP
↓
Residual Add
↓
output
```

公式：

$$
x_1
=
x+\operatorname{Attn}(N_1(x)),
$$

$$
\boxed{
y
=
x_1+\operatorname{MLP}(N_2(x_1))
}.
$$

本节也准备测试教学代码，但**母版不会运行任何 cell / test / experiment**。测试结果只有用户以后执行 work copy 后才能成立。

本轮完成即停，不进入 Step 15.1。


## 1. 一个 Block 为什么有两条 Residual Branches？

Block 内有两个不同职责的 sublayers。

### Attention branch

$$
x_1
=
x
+
\operatorname{Attn}(N_1(x)).
$$

Attention 负责：

> **cross-position information mixing**。

### MLP branch

$$
y
=
x_1
+
\operatorname{MLP}(N_2(x_1)).
$$

MLP 负责：

> **within-position nonlinear feature transformation**。

二者都返回 `(B,T,C)`，因此都能用 14.1 的 same-shape residual connection。

注意第二个 Norm / MLP 接收的是已经经过第一条 residual 更新后的 `x_1`，不是原始 x。


## 2. 把 Step 13 的 Multi-Head Attention 压缩成一个 Module

Attention 内部沿用已经学过的 shape chain：

```text
x                    (B,T,C)
↓ Linear(C,3C)
qkv                  (B,T,3C)
↓ split
q / k / v            (B,T,C)
↓ reshape C=H×D
q / k / v            (B,T,H,D)
↓ swap T/H
q / k / v            (B,H,T,D)
↓ QK^T / sqrt(D)
scores               (B,H,T,T)
↓ causal mask + softmax
weights              (B,H,T,T)
↓ weights @ V
head outputs         (B,H,T,D)
↓ swap back + merge
concatenated         (B,T,C)
↓ c_proj
attention output     (B,T,C)
```

Block 本节不引入新的 Attention 数学，只把 Step 13 的已学组件封装起来。


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        # Step 13.2 / 13.5 convention for this course:
        # QKV and output projections use bias=False.
        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        B, T, C = x.shape
        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)  # (B,H,T,D)
        v = mx.swapaxes(v, 1, 2)  # (B,H,T,D)

        k_t = mx.swapaxes(k, -1, -2)         # (B,H,D,T)
        scores = mx.matmul(q, k_t) * self.scale  # (B,H,T,T)

        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)  # (B,H,T,T)
        head_out = mx.matmul(weights, v)               # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)      # (B,T,H,D)
        concatenated = mx.reshape(token_major, (B, T, C))
        return self.c_proj(concatenated)                # (B,T,C)


# This cell only defines the module; it is not executed in the master notebook.


## 3. MLP 沿用 14.4：`C → 4C → GELU → C`

本节不改变 14.4 的设计：

```text
fc1 : C -> 4C, bias=True
GELU
fc2 : 4C -> C, bias=True
```

仍然是 position-wise shared function：对每个 token 独立使用同一套参数。

MLP 输入/输出都保持：

$$
(B,T,C).
$$


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(
            input_dims=dims,
            output_dims=4 * dims,
            bias=True,
        )
        self.act = nn.GELU()
        self.fc2 = nn.Linear(
            input_dims=4 * dims,
            output_dims=dims,
            bias=True,
        )

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


# Definition only; no runtime execution in this master notebook.


## 4. 真正组装一个 Pre-Norm Transformer Block

采用 Step 14.3 的 Pre-Norm skeleton：

$$
x_1
=
x+\operatorname{Attn}(N_1(x)),
$$

$$
y
=
x_1+\operatorname{MLP}(N_2(x_1)).
$$

因此需要：

```text
norm1
attention
norm2
mlp
```

两套 RMSNorm **不是同一个 Module 重复调用**，而是各自拥有独立 learned gamma。

原因是 Attention branch 与 MLP branch 可以学习不同的 feature scaling。


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(
            dims=dims,
            num_heads=num_heads,
        )

        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        # Pre-Norm Attention branch
        x = x + self.attn(self.norm1(x))

        # Pre-Norm MLP branch
        x = x + self.mlp(self.norm2(x))

        return x


# This is the first complete teaching Block definition.
# It is not executed in the master notebook.


## 5. Block 最重要的 Shape Invariants

给定：

$$
x:(B,T,C),
$$

一个 Block 应满足：

$$
\boxed{
Block(x):(B,T,C)
}.
$$

因此以下三件事都不能改变：

```text
B : batch size
T : sequence length
C : model width
```

特别是 roadmap 强调的：

$$
\boxed{T_{out}=T_{in}}.
$$

Attention 会在 positions 之间混合信息，但不会因为一次 Block forward 自动增加或删除 token positions。

MLP 也是 position-wise transform，同样保持 T。


In [ ]:
# TEST CELL — prepared only, NOT executed here.

B, T, C = 2, 5, 8
H = 2

block = TransformerBlock(dims=C, num_heads=H)
x = mx.zeros((B, T, C))
y = block(x)

assert y.shape == x.shape
assert y.shape[0] == B
assert y.shape[1] == T
assert y.shape[2] == C

print('input shape :', x.shape)
print('output shape:', y.shape)
print('shape invariants: expected to pass when this work copy is executed')

# No assertion has been run in the master notebook.


## 6. 手算一个 Block 的 Parameter Count

本节固定的实现选择：

### Attention

Fused QKV，bias=False：

$$
3C^2.
$$

`c_proj`，bias=False：

$$
C^2.
$$

所以 Attention projections：

$$
4C^2.
$$

### MLP

沿用 14.4 的 bias=True：

$$
8C^2+5C.
$$

### 两套 RMSNorm

每个只有 learned gamma：

$$
C.
$$

两套：

$$
2C.
$$

因此整个 Block：

$$
\boxed{
4C^2
+
(8C^2+5C)
+
2C
=
12C^2+7C
}.
$$

例如：

$$
C=8
$$

则：

$$
12(8^2)+7(8)
=
768+56
=
\boxed{824}.
$$

这里不包含 embedding、LM head、final norm 或其他 Block 之外的参数。


In [ ]:
# TEST CELL — prepared only, NOT executed here.

def numel(array):
    return math.prod(array.shape)


parameter_arrays = [
    block.norm1.weight,
    block.attn.qkv_proj.weight,
    block.attn.c_proj.weight,
    block.norm2.weight,
    block.mlp.fc1.weight,
    block.mlp.fc1.bias,
    block.mlp.fc2.weight,
    block.mlp.fc2.bias,
]

actual_parameter_count = sum(numel(p) for p in parameter_arrays)
expected_parameter_count = 12 * C * C + 7 * C

assert actual_parameter_count == expected_parameter_count

print('actual parameter count  :', actual_parameter_count)
print('expected parameter count:', expected_parameter_count)
print('for C=8, expected = 824')

# This code has not been executed in the master notebook.


## 7. Numerical Sanity Check 1：输出应该是 Finite

一次随机初始化 Block forward 后，最基础的 numerical sanity check 是：

> 输出不能出现 NaN / +inf / -inf。

MLX 0.32.2 提供：

```python
mx.isfinite(array)
```

然后可用：

```python
mx.all(...).item()
```

得到 Python boolean。

这个测试不能证明 Block “训练正确”或“模型质量好”；它只能排查最明显的 numerical failure。


In [ ]:
# TEST CELL — prepared only, NOT executed here.

finite_ok = mx.all(mx.isfinite(y)).item()
assert finite_ok

print('all output values finite:', finite_ok)

# Passing this test later would only establish a basic numerical sanity check.
# It would not prove training correctness or model quality.


## 8. Numerical Sanity Check 2：Future Token 不应改变 Earlier Outputs

这个测试直接检查当前 Block 的 **causal property**。

构造两个输入：

```text
x_a : 前 T-1 个 tokens 相同，最后 token = 0
x_b : 前 T-1 个 tokens 相同，最后 token 被大幅修改
```

对于位置：

$$
t<T-1,
$$

它们不允许看最后一个 future token。

而：

- RMSNorm 是 per-token；
- causal Attention 禁止 future visibility；
- MLP 是 position-wise；

所以数学上应有：

$$
\boxed{
Block(x_a)[:,:T-1,:]
\approx
Block(x_b)[:,:T-1,:]
}.
$$

这里用 `mx.allclose` 作为浮点近似比较。

这个 test 检查的是 **causal invariance**，不是要求最后一个 token 的输出一定改变。


In [ ]:
# TEST CELL — prepared only, NOT executed here.

B_test, T_test, C_test = 1, 4, C

prefix = mx.zeros((B_test, T_test - 1, C_test))
last_a = mx.zeros((B_test, 1, C_test))
last_b = mx.ones((B_test, 1, C_test)) * 100.0

x_a = mx.concatenate([prefix, last_a], axis=1)
x_b = mx.concatenate([prefix, last_b], axis=1)

y_a = block(x_a)
y_b = block(x_b)

earlier_a = y_a[:, : T_test - 1, :]
earlier_b = y_b[:, : T_test - 1, :]

causal_ok = mx.allclose(
    earlier_a,
    earlier_b,
    rtol=1e-5,
    atol=1e-6,
).item()

assert causal_ok
print('earlier positions unchanged by future-token edit:', causal_ok)

# This test is prepared but has NOT been executed in the master notebook.


## 9. 这些 Tests 能证明什么，不能证明什么？

如果用户以后执行 work copy 并全部通过，它们分别支持：

```text
shape test
-> Block preserves (B,T,C)

parameter-count test
-> implementation matches the intended module inventory

finite-value test
-> this forward pass has no obvious NaN/Inf

causal invariance test
-> changing a future token does not alter earlier outputs
```

但它们**不能**证明：

- optimizer / training loop 正确；
- gradient 数值一定健康；
- loss 一定下降；
- Block 能学会语言；
- 完整 GPT architecture 已经正确；
- 模型达到任何 quality benchmark。

因此本轮只称这些为 **prepared tests / static test logic**，不能称为“tests passed”。


## 10. 到这里，一个 Pre-Norm Block 的计算图已经完整

最终结构：

```text
x                         (B,T,C)
│
├──────── identity ───────────────────────┐
│                                         │
└─ RMSNorm -> Causal MHA -----------------┤
                                          ↓
                                         x1
                                          │
     ┌──────────────── identity ───────────┤
     │                                     │
     └─ RMSNorm -> MLP --------------------┤
                                           ↓
                                           y
                                        (B,T,C)
```

也就是：

$$
x_1=x+\operatorname{Attn}(N_1(x)),
$$

$$
y=x_1+\operatorname{MLP}(N_2(x_1)).
$$

这是第一次完整得到课程中的 repeated Transformer computation unit。

但本节仍然只停留在 **one Block**；还没有进入多个 Blocks 如何组成 GPT。


## 11. Step 14.5 材料边界

- 本节实现的是一个最小 Pre-Norm Transformer Block：`RMSNorm -> causal MHA -> residual -> RMSNorm -> MLP -> residual`。
- Attention 完整复用 Step 13 的 fused QKV、head split、batched causal attention、head merge 与 `c_proj`；没有引入新的 Attention 数学。
- Block 输入/输出均为 `(B,T,C)`；B、T、C shape invariants 都应保持，尤其 sequence length T 不变。
- 本节参数约定：Attention QKV/c_proj `bias=False`，MLP 两层 `bias=True`，两套 RMSNorm 各有 C 个 learned gamma，因此 Block 参数量是 `12C^2+7C`。
- 准备了 shape、parameter-count、finite-value 与 future-token causal-invariance tests，但本轮未运行任何 test；不能声称 tests passed。
- finite test 只是 basic numerical sanity check；causal test 只检查 earlier outputs 不依赖 future edit，均不能证明训练或模型质量正确。
- 本轮只准备 Step 14.5；不进入 Step 15.1，不开始 Transformer-to-GPT 组装、Block stacking、token embedding / positional path 或 GPT model implementation。
